# Phase 6 — Baseline ML Model: Proxy-Risk Classifier

**Purpose:** Train a multiclass classifier to predict `burden_tier_rel`
(the relative percentile-based stress tier) from *proxy socioeconomic features only*,
establishing a documented baseline before AutoML (Phase 7).

**Input:** `data/clean/clean_master_sa2_v2.csv`

**Outputs:**
- `outputs/models/baseline_rf.pkl` — Random Forest (used for SHAP in Phase 9)
- `outputs/models/baseline_logreg.pkl` — Logistic Regression
- `outputs/models/baseline_gb.pkl` — Gradient Boosting
- `outputs/models/label_encoder.pkl` — LabelEncoder for burden_tier_rel
- `outputs/figures/06_confusion_matrix.html`
- `outputs/figures/06_feature_importance.html`
- `outputs/figures/06_cv_macro_f1.html`

**ML framing:**
> "Can we identify high affordability-risk suburbs using public socioeconomic indicators
> alone — even where direct income data is unavailable?"

Features are SEIFA deprivation/advantage scores plus population and area.
The target is the **relative** burden tier (percentile-ranked within SA Water SA2s),
not the absolute threshold tier — so the classification problem is always well-defined
and class-balanced regardless of tariff level.

**Class balance:** 17 Critical / 25 High / 84 Moderate / 42 Low
Handled with `class_weight='balanced'` + stratified k-fold CV.

## Leakage lesson

An earlier version of this model included `median_hhd_inc_annual_adj` and
`estimated_annual_water_bill` as features. Since `burden_tier_rel` is derived from
`water_cost_burden_ratio = bill / income`, and the bill is a near-constant (same tariff
everywhere), income alone determines the tier. Adding income as a feature means the model
is learning to replicate the threshold formula — not solving a prediction problem.
The result was CV F1 ≈ 0.994 and test F1 = 1.000, which is not impressive; it is a
calculation check.

This version deliberately excludes income and bill from the feature set. The model predicts
stress risk from *publicly available socioeconomic proxies* only (SEIFA indexes).
A realistic F1 in the 0.5–0.8 range is expected and is more meaningful.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import pickle
import plotly.graph_objects as go
import plotly.express as px

from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    balanced_accuracy_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

ROOT   = Path('..').resolve()
CLEAN  = ROOT / 'data' / 'clean'
FIGS   = ROOT / 'outputs' / 'figures'
MODELS = ROOT / 'outputs' / 'models'
MODELS.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
print('Paths OK')

Paths OK


## 1. Load and prepare data

In [2]:
df_all = pd.read_csv(CLEAN / 'clean_master_sa2_v2.csv', dtype={'SA2_CODE21': str})
print('Loaded:', df_all.shape)
print()
print('burden_tier_rel distribution:')
print(df_all['burden_tier_rel'].value_counts())

Loaded: (176, 37)

burden_tier_rel distribution:
burden_tier_rel
Moderate    84
Low         42
High        25
Critical    17
Unknown      8
Name: count, dtype: int64


In [3]:
# Keep SA Water SA2s with valid burden ratios only
df = df_all[
    (df_all['provider_type'] == 'SA Water') &
    (df_all['burden_tier_rel'] != 'Unknown')
].copy()
print('After filtering to SA Water + known tiers:', df.shape)
print(df['burden_tier_rel'].value_counts())

After filtering to SA Water + known tiers: (168, 37)
burden_tier_rel
Moderate    84
Low         42
High        25
Critical    17
Name: count, dtype: int64


## 2. Feature selection

**Excluded (leakage):**
- `water_cost_burden_ratio`, all `burden_*` columns — these ARE the target or derived from it
- `median_hhd_inc_annual_adj`, `median_hhd_inc_*` — income determines the tier; including it
  would reduce the model to relearning the threshold formula
- `bill_*` columns — SA Water bill is nearly constant (only metro/country sewerage varies),
  so it contributes trivially
- `water_stress_index`, `stress_pct_rank`, `stress_tier` — derived from income/IER
- `estimated_hardship_need`, `hardship_priority_score` — downstream of tier assignment

**Included (genuine proxy features):**
- SEIFA indexes: irsd, irsad, ier, ieo — scores only (drop deciles; linear combination)
- `population`, `AREASQKM21` — size and density proxies

In [4]:
FEATURE_COLS = [
    'irsd_score',
    'irsad_score',
    'ier_score',
    'ieo_score',
    'population',
    'AREASQKM21',
]

TARGET_COL = 'burden_tier_rel'
tier_order = ['Low', 'Moderate', 'High', 'Critical']

X = df[FEATURE_COLS].copy()
y_raw = df[TARGET_COL].copy()

print('X shape:', X.shape)
print('Missing values per feature:')
print(X.isnull().sum())

X shape: (168, 6)
Missing values per feature:
irsd_score     3
irsad_score    3
ier_score      3
ieo_score      2
population     2
AREASQKM21     0
dtype: int64


In [5]:
le = LabelEncoder()
le.classes_ = np.array(tier_order)
y = le.transform(y_raw)

print('Label mapping:', dict(zip(le.classes_, le.transform(le.classes_))))
print('y distribution:', pd.Series(y).value_counts().sort_index())

Label mapping: {np.str_('Low'): np.int64(0), np.str_('Moderate'): np.int64(1), np.str_('High'): np.int64(2), np.str_('Critical'): np.int64(3)}
y distribution: 0    42
1    84
2    25
3    17
Name: count, dtype: int64


## 3. Train / test split

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

print('Train:', X_train.shape, '| Test:', X_test.shape)
print('\nTrain class distribution:')
print(pd.Series(y_train).value_counts().sort_index())
print('\nTest class distribution:')
print(pd.Series(y_test).value_counts().sort_index())

Train: (134, 6) | Test: (34, 6)

Train class distribution:
0    33
1    67
2    20
3    14
Name: count, dtype: int64

Test class distribution:


0     9
1    17
2     5
3     3
Name: count, dtype: int64


## 4. Define models

All models use `class_weight='balanced'` to up-weight the minority Critical class.
Pipelines include `SimpleImputer(median)` for the SA2s with missing SEIFA scores.

In [7]:
models = {
    'LogReg': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
        ('clf',     LogisticRegression(
            class_weight='balanced',
            max_iter=1000,
            random_state=RANDOM_STATE,
            C=1.0,
        )),
    ]),
    'RandomForest': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('clf',     RandomForestClassifier(
            n_estimators=300,
            class_weight='balanced',
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ]),
    'GradientBoosting': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('clf',     GradientBoostingClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=3,
            random_state=RANDOM_STATE,
        )),
    ]),
}

print('Models defined:', list(models.keys()))

Models defined: ['LogReg', 'RandomForest', 'GradientBoosting']


## 5. Cross-validation (StratifiedKFold, n=5)

Primary metric: macro F1 (treats all classes equally regardless of size).
Expect realistic scores 0.5–0.8; SEIFA scores partially but imperfectly predict income quartiles.

In [8]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_results = {}
for name, model in models.items():
    scores = cross_validate(
        model, X_train, y_train,
        cv=cv,
        scoring={'macro_f1': 'f1_macro', 'balanced_acc': 'balanced_accuracy'},
        return_train_score=False,
    )
    cv_results[name] = {
        'macro_f1_mean': scores['test_macro_f1'].mean(),
        'macro_f1_std':  scores['test_macro_f1'].std(),
        'balanced_acc_mean': scores['test_balanced_acc'].mean(),
        'fold_f1': scores['test_macro_f1'],
    }
    print(f'{name}: macro_F1 = {scores["test_macro_f1"].mean():.3f}'
          f' ± {scores["test_macro_f1"].std():.3f}'
          f' | balanced_acc = {scores["test_balanced_acc"].mean():.3f}')

LogReg: macro_F1 = 0.616 ± 0.070 | balanced_acc = 0.654


RandomForest: macro_F1 = 0.592 ± 0.102 | balanced_acc = 0.598


GradientBoosting: macro_F1 = 0.615 ± 0.106 | balanced_acc = 0.619


In [9]:
fig = go.Figure()
colors = {'LogReg': '#636EFA', 'RandomForest': '#00CC96', 'GradientBoosting': '#EF553B'}

for name, res in cv_results.items():
    fig.add_trace(go.Box(
        y=res['fold_f1'],
        name=name,
        marker_color=colors[name],
        boxmean=True,
    ))

fig.update_layout(
    title='5-Fold CV Macro F1 by Model — Proxy-Risk Classifier (SEIFA features only)',
    yaxis_title='Macro F1',
    yaxis=dict(range=[0, 1]),
    template='plotly_white',
    width=700, height=450,
)
fig.write_html(str(FIGS / '06_cv_macro_f1.html'))
print('Saved 06_cv_macro_f1.html')

Saved 06_cv_macro_f1.html


## 6. Train final models on full training set

In [10]:
fitted_models = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    fitted_models[name] = model
    print(f'{name} trained.')

LogReg trained.


RandomForest trained.


GradientBoosting trained.


## 7. Evaluate on held-out test set

In [11]:
test_results = {}
for name, model in fitted_models.items():
    y_pred = model.predict(X_test)
    macro_f1 = f1_score(y_test, y_pred, average='macro')
    bal_acc  = balanced_accuracy_score(y_test, y_pred)
    test_results[name] = {'y_pred': y_pred, 'macro_f1': macro_f1, 'balanced_acc': bal_acc}
    print(f'\n=== {name} ===')
    print(f'Test macro F1: {macro_f1:.3f} | Balanced accuracy: {bal_acc:.3f}')
    print(classification_report(y_test, y_pred, target_names=tier_order))


=== LogReg ===
Test macro F1: 0.563 | Balanced accuracy: 0.588
              precision    recall  f1-score   support

         Low       0.88      0.78      0.82         9
    Moderate       0.75      0.71      0.73        17
        High       0.20      0.20      0.20         5
    Critical       0.40      0.67      0.50         3

    accuracy                           0.65        34
   macro avg       0.56      0.59      0.56        34
weighted avg       0.67      0.65      0.66        34


=== RandomForest ===
Test macro F1: 0.618 | Balanced accuracy: 0.604
              precision    recall  f1-score   support

         Low       0.86      0.67      0.75         9
    Moderate       0.68      0.88      0.77        17
        High       0.50      0.20      0.29         5
    Critical       0.67      0.67      0.67         3

    accuracy                           0.71        34
   macro avg       0.68      0.60      0.62        34
weighted avg       0.70      0.71      0.68        

In [12]:
summary = pd.DataFrame([
    {
        'Model': name,
        'CV Macro F1': f"{cv_results[name]['macro_f1_mean']:.3f} ± {cv_results[name]['macro_f1_std']:.3f}",
        'Test Macro F1': f"{res['macro_f1']:.3f}",
        'Test Balanced Acc': f"{res['balanced_acc']:.3f}",
    }
    for name, res in test_results.items()
])
print(summary.to_string(index=False))

           Model   CV Macro F1 Test Macro F1 Test Balanced Acc
          LogReg 0.616 ± 0.070         0.563             0.588
    RandomForest 0.592 ± 0.102         0.618             0.604
GradientBoosting 0.615 ± 0.106         0.578             0.604


## 8. Confusion matrix — best model by CV macro F1

In [13]:
best_name = max(cv_results, key=lambda k: cv_results[k]['macro_f1_mean'])
print(f'Best model by CV macro F1: {best_name} ({cv_results[best_name]["macro_f1_mean"]:.3f})')

y_pred_best = test_results[best_name]['y_pred']
cm = confusion_matrix(y_test, y_pred_best)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig = go.Figure(data=go.Heatmap(
    z=cm_norm,
    x=[f'Pred: {t}' for t in tier_order],
    y=[f'True: {t}' for t in tier_order],
    colorscale='Blues',
    zmin=0, zmax=1,
    text=[[f'{cm[i][j]}<br>({cm_norm[i][j]:.0%})' for j in range(4)] for i in range(4)],
    texttemplate='%{text}',
))
fig.update_layout(
    title=f'Confusion Matrix — {best_name} (row-normalised recall)',
    xaxis_title='Predicted',
    yaxis_title='Actual',
    template='plotly_white',
    width=600, height=500,
)
fig.write_html(str(FIGS / '06_confusion_matrix.html'))
print('Saved 06_confusion_matrix.html')

Best model by CV macro F1: LogReg (0.616)
Saved 06_confusion_matrix.html


## 9. Feature importance — Random Forest

In [14]:
rf_clf = fitted_models['RandomForest']['clf']
importances = pd.Series(
    rf_clf.feature_importances_,
    index=FEATURE_COLS,
).sort_values(ascending=True)

fig = go.Figure(go.Bar(
    x=importances.values,
    y=importances.index,
    orientation='h',
    marker_color='#00CC96',
))
fig.update_layout(
    title='Random Forest — Feature Importance (MDI) — SEIFA proxy-risk model',
    xaxis_title='Mean Decrease in Impurity',
    template='plotly_white',
    width=700, height=400,
)
fig.write_html(str(FIGS / '06_feature_importance.html'))
print('Saved 06_feature_importance.html')
print()
print('Feature importances:')
print(importances.sort_values(ascending=False).round(4))

Saved 06_feature_importance.html

Feature importances:
irsd_score     0.2306
irsad_score    0.2030
ier_score      0.1617
AREASQKM21     0.1392
ieo_score      0.1350
population     0.1305
dtype: float64


## 10. Attach predictions to dataset

In [15]:
best_model = fitted_models['RandomForest']  # Use RF for SHAP compatibility (Phase 9)
X_all = df[FEATURE_COLS].copy()
df['predicted_tier_encoded'] = best_model.predict(X_all)
df['predicted_tier']         = le.inverse_transform(df['predicted_tier_encoded'])
df['predicted_proba_critical'] = best_model.predict_proba(X_all)[:, tier_order.index('Critical')]

df['tier_match'] = df['burden_tier_rel'] == df['predicted_tier']
print(f'Overall accuracy on training SA2s: {df["tier_match"].mean():.1%}')
print()
print('Predicted vs actual Critical SA2s (relative tier):')
print(df[df['burden_tier_rel'] == 'Critical'][[
    'SA2_NAME21', 'burden_tier_rel', 'burden_tier_abs', 'predicted_tier',
    'predicted_proba_critical', 'water_cost_burden_ratio'
]].sort_values('predicted_proba_critical', ascending=False).to_string(index=False))

Overall accuracy on training SA2s: 94.0%

Predicted vs actual Critical SA2s (relative tier):
                     SA2_NAME21 burden_tier_rel burden_tier_abs predicted_tier  predicted_proba_critical  water_cost_burden_ratio
                        Renmark        Critical        Moderate       Critical                  0.876667                 0.022241
                          Berri        Critical        Moderate       Critical                  0.870000                 0.021770
                       Wallaroo        Critical        Moderate       Critical                  0.836667                 0.024693
                      Millicent        Critical        Moderate       Critical                  0.766667                 0.022894
                       Lonsdale        Critical        Moderate       Critical                  0.766667                 0.026936
Peterborough - Mount Remarkable        Critical        Moderate       Critical                  0.753333                 0.0242

## 11. Save models and encoder

In [16]:
with open(MODELS / 'baseline_rf.pkl', 'wb') as f:
    pickle.dump(fitted_models['RandomForest'], f)

with open(MODELS / 'baseline_logreg.pkl', 'wb') as f:
    pickle.dump(fitted_models['LogReg'], f)

with open(MODELS / 'baseline_gb.pkl', 'wb') as f:
    pickle.dump(fitted_models['GradientBoosting'], f)

with open(MODELS / 'label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

# Save feature column list so Phase 9 (SHAP) uses the same set
import json
with open(MODELS / 'feature_cols.json', 'w') as f:
    json.dump(FEATURE_COLS, f)

print('Saved models:')
for p in sorted(MODELS.glob('*.pkl')):
    print(f'  {p.name}')
print(f'  feature_cols.json')

Saved models:
  baseline_gb.pkl
  baseline_logreg.pkl
  baseline_rf.pkl
  label_encoder.pkl
  pycaret_best_model.pkl
  feature_cols.json


## 12. Phase summary

**Model:** Proxy-risk classifier — predicts `burden_tier_rel` from SEIFA indexes only.  
**Features:** irsd_score, irsad_score, ier_score, ieo_score, population, AREASQKM21  
**Target:** burden_tier_rel (relative percentile tier within SA Water SA2s)  

**Results stored in outputs/models/:**
- `baseline_rf.pkl` — used for SHAP (Phase 9); TreeExplainer requires RF not GB
- `label_encoder.pkl`, `feature_cols.json`

**Leakage summary:**  
Previous version included income/bill → F1 = 1.000 (formula replication, not prediction).  
This version uses SEIFA proxies → realistic F1 shows genuine predictive signal from deprivation indexes.

**Portfolio framing:**  
Lead with the affordability analysis (deterministic formula + simulation).  
Present ML as: *"SEIFA socioeconomic indexes can identify roughly X% of high-risk suburbs
without income data — useful for targeting outreach in areas where income data is unavailable."*

**Next:** Phase 7 — AutoML with PyCaret to benchmark against this baseline.  
PyCaret `setup()` resets the index — save `SA2_CODE21` before calling `setup()`.